# Workflow YAML Validation

This notebook validates SilkRoute workflow YAML descriptors without making API calls. It is safe to run offline as long as SilkRoute and its local dependencies are importable.

The examples use repository-relative paths and the frozen `workflow-v1` schema.

In [ ]:
from pathlib import Path
from pprint import pprint

from silkroute.cli.workflows import load_workflow_recipe, validate_workflow_recipe

repo_root = Path.cwd().parent.parent
workflow_dir = repo_root / "examples" / "workflows"
invalid_dir = workflow_dir / "invalid"

print(workflow_dir)

## Validate workflow-v1 examples

The valid descriptors under `examples/workflows/` should include `schema_version: "workflow-v1"` and validate without executing any APIs.

In [ ]:
valid_paths = sorted(path for path in workflow_dir.glob("*.yml"))
validated = {}

for config_path in valid_paths:
    recipe = load_workflow_recipe(config_path)
    validated[config_path.name] = validate_workflow_recipe(recipe)

print(f"Validated {len(validated)} workflow descriptors:")
pprint(list(validated))

## Inspect normalized values

Validation normalizes executable values while preserving descriptive metadata.

In [ ]:
example_name = "protein_query_first_minimal.yml"
normalized = validated[example_name]

pprint({
    "schema_version": normalized["schema_version"],
    "query": normalized["query"],
    "modality": normalized["modality"],
    "mode": normalized["mode"],
    "export_format": normalized["export_format"],
    "output": normalized["output"],
})

## Validate intentionally invalid examples

The files under `examples/workflows/invalid/` are expected to fail validation. These examples demonstrate common schema errors without making API calls.

In [ ]:
invalid_results = {}

for config_path in sorted(invalid_dir.glob("*.yml")):
    try:
        recipe = load_workflow_recipe(config_path)
        validate_workflow_recipe(recipe)
    except (TypeError, ValueError) as exc:
        invalid_results[config_path.name] = str(exc)
    else:
        invalid_results[config_path.name] = "Unexpectedly valid"

pprint(invalid_results)